# Chapter 9: AI Sidecar for a Mock CRM

**AI Enterprise Architecture — Companion Notebook**

The **sidecar pattern** adds new capabilities alongside an existing system *without modifying it*. In microservices, sidecars handle logging, auth, or service-mesh proxying. Here we build an **AI sidecar** for a mock CRM database that:

1. **Reads** customer records and interaction notes (read-only).
2. **Classifies** the sentiment of each interaction using an LLM.
3. **Generates** per-account health summaries.
4. **Surfaces** at-risk accounts based on negative sentiment trends.

The CRM (pandas DataFrames) is never modified — the sidecar only reads.

> **Key insight:** Sidecars add AI value with zero changes to existing systems.

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────
!pip install -q openai pandas tabulate

### API Key Setup

The sidecar uses OpenAI for sentiment classification and summary generation. Set your key as an environment variable:

```python
import os
os.environ["OPENAI_API_KEY"] = "sk-..."
```

Or use the Colab Secrets panel (key icon in the left sidebar).

In [ ]:
import os
import json
import pandas as pd
from datetime import date, timedelta
from tabulate import tabulate

import openai

OPENAI_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_KEY:
    print("WARNING: OPENAI_API_KEY not set. Set it before running the sidecar.")
else:
    print("OpenAI API key detected.")

client = openai.OpenAI(api_key=OPENAI_KEY) if OPENAI_KEY else None

---
## Step 1 — Create the Mock CRM Database

We define two tables as pandas DataFrames:

- **customers** — id, name, company, tier (gold / silver / bronze), since_date
- **interactions** — id, customer_id, date, channel, notes

Eight customers, each with 3-5 interactions. The notes are realistic snippets that cover a range of sentiments.

In [ ]:
# ── Customers Table ──────────────────────────────────────────────────

customers = pd.DataFrame([
    {"id": 1, "name": "Alice Nguyen",    "company": "Pinnacle Corp",    "tier": "gold",   "since_date": "2021-03-15"},
    {"id": 2, "name": "Bob Martinez",    "company": "Vertex Inc",       "tier": "silver", "since_date": "2022-07-01"},
    {"id": 3, "name": "Carol Zhang",     "company": "NovaTech",         "tier": "gold",   "since_date": "2020-11-20"},
    {"id": 4, "name": "Dan Okafor",      "company": "Meridian LLC",     "tier": "bronze", "since_date": "2023-01-10"},
    {"id": 5, "name": "Eva Petrov",      "company": "Cirrus Systems",   "tier": "silver", "since_date": "2022-04-08"},
    {"id": 6, "name": "Frank Dubois",    "company": "Atlas Group",      "tier": "gold",   "since_date": "2019-09-30"},
    {"id": 7, "name": "Grace Kim",       "company": "Horizon Digital",  "tier": "bronze", "since_date": "2023-06-22"},
    {"id": 8, "name": "Hiro Tanaka",     "company": "Summit Analytics", "tier": "silver", "since_date": "2021-12-05"},
])

print(f"Customers: {len(customers)}")
customers

In [ ]:
# ── Interactions Table ────────────────────────────────────────────────
# Each customer has 3-5 interactions with realistic notes.

interactions = pd.DataFrame([
    # Alice Nguyen — mostly happy, one concern
    {"id": 101, "customer_id": 1, "date": "2025-12-01", "channel": "email",  "notes": "Loved the new dashboard update. Much faster than before."},
    {"id": 102, "customer_id": 1, "date": "2026-01-10", "channel": "call",   "notes": "Asked about enterprise SSO integration. Very interested."},
    {"id": 103, "customer_id": 1, "date": "2026-02-15", "channel": "chat",   "notes": "Reported a minor UI glitch on the reports page. Not urgent."},

    # Bob Martinez — mixed, trending negative
    {"id": 201, "customer_id": 2, "date": "2025-11-05", "channel": "email",  "notes": "Happy with onboarding. Team is ramping up quickly."},
    {"id": 202, "customer_id": 2, "date": "2026-01-20", "channel": "call",   "notes": "Frustrated with API rate limits. Says it blocks their batch jobs."},
    {"id": 203, "customer_id": 2, "date": "2026-02-28", "channel": "email",  "notes": "Escalated to their VP. Threatening to switch vendors if rate limits are not fixed."},
    {"id": 204, "customer_id": 2, "date": "2026-03-05", "channel": "call",   "notes": "Still unhappy. Wants a meeting with our engineering lead this week."},

    # Carol Zhang — consistently positive
    {"id": 301, "customer_id": 3, "date": "2025-10-12", "channel": "email",  "notes": "Renewed for 2 more years. Very satisfied with support quality."},
    {"id": 302, "customer_id": 3, "date": "2026-01-08", "channel": "chat",   "notes": "Requested a case study collaboration. Wants to co-present at conference."},
    {"id": 303, "customer_id": 3, "date": "2026-02-20", "channel": "email",  "notes": "Referred two new prospects to us. Excellent advocate."},

    # Dan Okafor — neutral to slightly negative
    {"id": 401, "customer_id": 4, "date": "2025-12-15", "channel": "call",   "notes": "Initial setup complete. No issues reported so far."},
    {"id": 402, "customer_id": 4, "date": "2026-01-25", "channel": "email",  "notes": "Confused by the new pricing tier. Asked for a breakdown."},
    {"id": 403, "customer_id": 4, "date": "2026-03-01", "channel": "chat",   "notes": "Says the documentation is outdated for the v3 API. Wasted half a day."},

    # Eva Petrov — positive with one blip
    {"id": 501, "customer_id": 5, "date": "2025-11-20", "channel": "email",  "notes": "Great onboarding experience. Impressed with the speed."},
    {"id": 502, "customer_id": 5, "date": "2026-01-05", "channel": "call",   "notes": "Had a billing discrepancy. Resolved quickly by support."},
    {"id": 503, "customer_id": 5, "date": "2026-02-10", "channel": "email",  "notes": "Expanding usage to their London office. Very positive."},
    {"id": 504, "customer_id": 5, "date": "2026-03-02", "channel": "chat",   "notes": "Asking about volume discounts. Planning to double seats."},

    # Frank Dubois — declining, at risk
    {"id": 601, "customer_id": 6, "date": "2025-09-10", "channel": "email",  "notes": "Long-time customer. Always been satisfied with the platform."},
    {"id": 602, "customer_id": 6, "date": "2025-12-18", "channel": "call",   "notes": "Complained about recent downtime. Affected a client demo."},
    {"id": 603, "customer_id": 6, "date": "2026-02-05", "channel": "email",  "notes": "Says competitor X now offers the same features at 30% less cost."},
    {"id": 604, "customer_id": 6, "date": "2026-03-03", "channel": "call",   "notes": "Asked about contract cancellation terms. Renewal is in 60 days."},

    # Grace Kim — new, neutral
    {"id": 701, "customer_id": 7, "date": "2026-01-15", "channel": "email",  "notes": "Just signed up. Setting up their dev environment."},
    {"id": 702, "customer_id": 7, "date": "2026-02-01", "channel": "chat",   "notes": "Had a question about data export formats. Resolved in chat."},
    {"id": 703, "customer_id": 7, "date": "2026-03-06", "channel": "email",  "notes": "Positive feedback on the quick-start guide. Found it helpful."},

    # Hiro Tanaka — mixed signals
    {"id": 801, "customer_id": 8, "date": "2025-12-20", "channel": "call",   "notes": "Praised the analytics module. Uses it daily for reporting."},
    {"id": 802, "customer_id": 8, "date": "2026-01-30", "channel": "email",  "notes": "Reported a data sync delay with their Salesforce integration."},
    {"id": 803, "customer_id": 8, "date": "2026-02-25", "channel": "chat",   "notes": "Sync issue still not resolved. Getting impatient."},
    {"id": 804, "customer_id": 8, "date": "2026-03-07", "channel": "call",   "notes": "Sync fixed. Happy again but wants assurance it won't recur."},
])

print(f"Interactions: {len(interactions)}")
interactions.head(10)

---
## Step 2 — Build the AI Sidecar

The `AISidecar` class encapsulates all AI-powered analysis. It receives the CRM DataFrames **by reference** (read-only) and writes results to its own internal state.

In [ ]:
class AISidecar:
    """AI Sidecar that reads from a CRM (DataFrames) without modifying them.

    Capabilities:
      - classify_sentiments()  : label each interaction as positive/neutral/negative
      - generate_summaries()   : produce a health summary per customer
      - identify_at_risk()     : flag accounts with negative sentiment trends
    """

    def __init__(self, customers_df: pd.DataFrame, interactions_df: pd.DataFrame, openai_client):
        # Read-only references to CRM data
        self._customers = customers_df
        self._interactions = interactions_df
        self._client = openai_client

        # Sidecar's own state (never written back to CRM)
        self.sentiments: pd.DataFrame = pd.DataFrame()
        self.summaries: dict = {}        # customer_id -> summary string
        self.at_risk: list = []          # list of customer_ids

    # ── LLM helper ───────────────────────────────────────────────────

    def _ask(self, system: str, user: str) -> str:
        """Send a chat completion request and return the assistant's text."""
        resp = self._client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
            max_tokens=256,
            temperature=0,
        )
        return resp.choices[0].message.content.strip()

    # ── 1. Sentiment Classification ──────────────────────────────────

    def classify_sentiments(self) -> pd.DataFrame:
        """Classify each interaction note as positive, neutral, or negative."""
        system_prompt = (
            "You are a sentiment classifier. Given a customer interaction note, "
            "respond with exactly one word: positive, neutral, or negative."
        )
        rows = []
        for _, row in self._interactions.iterrows():
            sentiment = self._ask(system_prompt, row["notes"]).lower()
            # Normalise unexpected responses
            if sentiment not in ("positive", "neutral", "negative"):
                sentiment = "neutral"
            rows.append({"interaction_id": row["id"], "customer_id": row["customer_id"],
                         "date": row["date"], "sentiment": sentiment})
            print(f"  [sentiment] interaction {row['id']:>3d} -> {sentiment}")

        self.sentiments = pd.DataFrame(rows)
        return self.sentiments

    # ── 2. Account Health Summaries ──────────────────────────────────

    def generate_summaries(self) -> dict:
        """Generate a 2-3 sentence health summary for each customer."""
        system_prompt = (
            "You are a customer success analyst. Given a customer profile and their "
            "recent interactions with sentiment labels, write a concise 2-3 sentence "
            "account health summary. Mention the overall trend and any action items."
        )
        for _, cust in self._customers.iterrows():
            cid = cust["id"]
            # Gather interactions + sentiments for this customer
            cust_interactions = self._interactions[self._interactions["customer_id"] == cid]
            cust_sentiments = self.sentiments[self.sentiments["customer_id"] == cid]
            merged = cust_interactions.merge(cust_sentiments[["interaction_id", "sentiment"]],
                                             left_on="id", right_on="interaction_id")

            # Build a text block for the LLM
            interaction_lines = []
            for _, ix in merged.iterrows():
                interaction_lines.append(f"- [{ix['date']}] ({ix['sentiment']}) {ix['notes']}")

            user_msg = (
                f"Customer: {cust['name']} | Company: {cust['company']} | "
                f"Tier: {cust['tier']} | Since: {cust['since_date']}\n\n"
                f"Recent interactions:\n" + "\n".join(interaction_lines)
            )

            summary = self._ask(system_prompt, user_msg)
            self.summaries[cid] = summary
            print(f"  [summary] {cust['name']} done")

        return self.summaries

    # ── 3. At-Risk Detection ─────────────────────────────────────────

    def identify_at_risk(self) -> list:
        """Flag accounts whose most recent interactions trend negative.

        Heuristic: if the last 2 interactions include at least one 'negative'
        sentiment, the account is considered at risk.
        """
        self.at_risk = []
        for cid in self._customers["id"]:
            cust_sent = self.sentiments[self.sentiments["customer_id"] == cid].sort_values("date")
            if len(cust_sent) < 2:
                continue
            recent = cust_sent.tail(2)["sentiment"].tolist()
            if "negative" in recent:
                self.at_risk.append(cid)
        return self.at_risk

---
## Step 3 — Run the Sidecar Analysis

We instantiate the sidecar and run all three stages sequentially.

In [ ]:
if client is None:
    print("Skipping — set OPENAI_API_KEY to run the sidecar.")
else:
    sidecar = AISidecar(customers, interactions, client)

    print("=" * 50)
    print("Stage 1: Classifying sentiments...")
    print("=" * 50)
    sidecar.classify_sentiments()

    print("\n" + "=" * 50)
    print("Stage 2: Generating account summaries...")
    print("=" * 50)
    sidecar.generate_summaries()

    print("\n" + "=" * 50)
    print("Stage 3: Identifying at-risk accounts...")
    print("=" * 50)
    sidecar.identify_at_risk()
    print("Done.")

---
## Step 4 — Display Results

### 4.1 Sentiment-Annotated Interactions

In [ ]:
if client is not None:
    # Merge interactions with sentiment labels for display
    annotated = interactions.merge(
        sidecar.sentiments[["interaction_id", "sentiment"]],
        left_on="id", right_on="interaction_id",
    ).merge(
        customers[["id", "name"]], left_on="customer_id", right_on="id", suffixes=("", "_cust")
    )

    display_cols = ["name", "date", "channel", "notes", "sentiment"]
    # Truncate notes for readability
    annotated["notes"] = annotated["notes"].str[:60] + "..."
    print(tabulate(annotated[display_cols], headers="keys", tablefmt="github", showindex=False))

### 4.2 Account Health Summaries

In [ ]:
if client is not None:
    for cid, summary in sidecar.summaries.items():
        cust = customers[customers["id"] == cid].iloc[0]
        print(f"\n{'─' * 60}")
        print(f"{cust['name']} ({cust['company']}) — {cust['tier'].upper()} tier")
        print(f"{'─' * 60}")
        print(summary)

### 4.3 Risk Dashboard

In [ ]:
if client is not None:
    print("=" * 60)
    print("AT-RISK ACCOUNTS")
    print("=" * 60)

    if not sidecar.at_risk:
        print("No at-risk accounts detected.")
    else:
        risk_rows = []
        for cid in sidecar.at_risk:
            cust = customers[customers["id"] == cid].iloc[0]
            recent_sents = sidecar.sentiments[
                sidecar.sentiments["customer_id"] == cid
            ].sort_values("date").tail(3)["sentiment"].tolist()
            risk_rows.append({
                "customer": cust["name"],
                "company": cust["company"],
                "tier": cust["tier"],
                "recent_sentiments": " -> ".join(recent_sents),
            })
        print(tabulate(pd.DataFrame(risk_rows), headers="keys", tablefmt="github", showindex=False))
        print(f"\n{len(sidecar.at_risk)} account(s) flagged. Recommend immediate CSM outreach.")

### 4.4 Verify CRM Was Not Modified

A core guarantee of the sidecar pattern: the source data is untouched.

In [ ]:
# ── Verify CRM integrity ─────────────────────────────────────────────
# The original DataFrames should have no extra columns or modified rows.

assert list(customers.columns) == ["id", "name", "company", "tier", "since_date"], \
    "Customers table was modified!"
assert list(interactions.columns) == ["id", "customer_id", "date", "channel", "notes"], \
    "Interactions table was modified!"
assert len(customers) == 8, "Customer count changed!"
assert len(interactions) == 30, "Interaction count changed!"

print("CRM integrity check PASSED.")
print(f"  customers:    {len(customers)} rows, columns = {list(customers.columns)}")
print(f"  interactions: {len(interactions)} rows, columns = {list(interactions.columns)}")
print("\nThe sidecar added AI capabilities without modifying the CRM.")

---
## Takeaways

| Concept | Implementation | Enterprise Parallel |
|---|---|---|
| Read-only access | Sidecar reads DataFrames, never writes | Service mesh proxy (Envoy) observes traffic without altering it |
| Sentiment classification | LLM labels each interaction | NLP pipeline in a data lakehouse |
| Account summaries | LLM synthesises per-customer narrative | Executive dashboards generated from telemetry |
| Risk detection | Heuristic over sentiment trend | SRE alerting on error-rate trends |

**Key insight:** Sidecars add AI value with zero changes to existing systems. This is the lowest-risk, highest-ROI pattern for your first AI integration with a legacy system.